# Sistema Multi-Agente de Soporte Interno con RAG

Este notebook implementa un sistema automatizado de soporte interno empresarial utilizando múltiples agentes especializados, Retrieval-Augmented Generation (RAG), y orquestación con LangGraph.

## Estructura del Notebook

1. **Setup e Imports** - Configuración inicial y dependencias
2. **Carga de Documentos y Vector Stores** - Preparación de bases de conocimiento
3. **Definición de Agentes** - Agentes especializados RRH, IT, Finance
4. **Orquestador y Enrutamiento** - Router y grafo LangGraph
5. **Pruebas y Ejemplos** - Ejecución de consultas de prueba
6. **Integración con Langfuse** - Workflow tracing y evaluación

---

## 1. Setup e Imports

In [ ]:
# %pip install langchain langchain-openai langchain-community langgraph chromadb python-dotenv sentence-transformers tiktoken langfuse

In [ ]:
import os
import sys
from pathlib import Path

# Agregar el directorio raíz al path
project_root = Path(os.getcwd())
sys.path.insert(0, str(project_root))

from dotenv import load_dotenv

# Cargar configuración
load_dotenv()

print("✅ Setup completado")

---

## 2. Carga de Documentos y Vector Stores

In [ ]:
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

def load_documents_from_folder(folder_path, glob_pattern="**/*.md"):
    """Carga documentos desde una carpeta."""
    loader = DirectoryLoader(folder_path, glob=glob_pattern)
    return loader.load()

def create_vector_store(documents, collection_name):
    """Crea un vector store con los documentos."""
    # Configurar embeddings
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        model_kwargs={'device': 'cpu'}
    )
    
    # Crear vector store
    vectorstore = Chroma.from_documents(
        documents=documents,
        embedding=embeddings,
        collection_name=collection_name,
        persist_directory="./knowledge_bases/"
    )
    return vectorstore

print("✅ Funciones de carga definidas")

In [ ]:
# Cargar documentos de cada dominio
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Configurar splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

# Cargar documentos HR
hr_loader = DirectoryLoader("./data/hr_docs/", glob="**/*")
hr_docs = hr_loader.load()
hr_chunks = text_splitter.split_documents(hr_docs)
print(f"📂 HR: {len(hr_docs)} documentos, {len(hr_chunks)} chunks")

# Cargar documentos Tech
tech_loader = DirectoryLoader("./data/tech_docs/", glob="**/*")
tech_docs = tech_loader.load()
tech_chunks = text_splitter.split_documents(tech_docs)
print(f"📂 Tech: {len(tech_docs)} documentos, {len(tech_chunks)} chunks")

# Cargar documentos Finance
finance_loader = DirectoryLoader("./data/finance_docs/", glob="**/*")
finance_docs = finance_loader.load()
finance_chunks = text_splitter.split_documents(finance_docs)
print(f"📂 Finance: {len(finance_docs)} documentos, {len(finance_chunks)} chunks")

print(f"\n✅ Total: {len(hr_chunks) + len(tech_chunks) + len(finance_chunks)} chunks")

In [ ]:
# Crear vector stores por dominio
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={'device': 'cpu'}
)

# HR Vector Store
hr_vectorstore = Chroma.from_documents(
    documents=hr_chunks,
    embedding=embeddings,
    collection_name="hr_knowledge",
    persist_directory="./knowledge_bases/hr_vector_db"
)

# Tech Vector Store
tech_vectorstore = Chroma.from_documents(
    documents=tech_chunks,
    embedding=embeddings,
    collection_name="tech_knowledge",
    persist_directory="./knowledge_bases/tech_vector_db"
)

# Finance Vector Store
finance_vectorstore = Chroma.from_documents(
    documents=finance_chunks,
    embedding=embeddings,
    collection_name="finance_knowledge",
    persist_directory="./knowledge_bases/finance_vector_db"
)

print("✅ Vector stores creados y persistidos")

---

## 3. Definición de Agentes

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import Runnable

# Configuración del LLM
llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0.1,
    api_key=os.getenv("OPENAI_API_KEY", "sk-dummy")
)

print("✅ LLM configurado")

In [ ]:
class RRHAgent(Runnable):
    """Agente de Recursos Humanos con RAG."""
    
    def __init__(self, llm, vectorstore):
        self.llm = llm
        self.vectorstore = vectorstore
        
        # Few-shot prompts
        self.system_prompt = """Eres el agente de Recursos Humanos de la empresa.
        Tu objetivo es ayudar a los empleados con sus consultas sobre políticas de RRHH.
        
        EJEMPLOS:
        - Consulta: Cuántos días de vacaciones tengo? Respuesta: Según tu antigüedad...
        - Consulta: Puedo trabajar remoto? Respuesta: La política permite hasta 2 días...
        
        Responde siempre de manera clara, concisa y amigable.
        Si no tienes información suficiente, pide clarificación."""
    
    def invoke(self, input_data, config=None):
        query = input_data if isinstance(input_data, str) else input_data.get("query", "")
        
        # Retrieve context
        docs = self.vectorstore.similarity_search(query, k=4)
        context = "\n\n".join([d.page_content for d in docs])
        
        # Create prompt
        prompt = ChatPromptTemplate.from_messages([
            ("system", self.system_prompt),
            ("human", f"Contexto:\n{context}\n\nConsulta: {query}")
        ])
        
        # Get response
        chain = prompt | self.llm
        return {"response": chain.invoke({}).content, "context": context}

class ITAgent(Runnable):
    """Agente de Soporte Técnico con RAG."""
    
    def __init__(self, llm, vectorstore):
        self.llm = llm
        self.vectorstore = vectorstore
        
        self.system_prompt = """Eres el agente de Soporte Técnico de la empresa.
        Tu objetivo es resolver problemas técnicos de los empleados.
        
        EJEMPLOS:
        - Consulta: Mi laptop no enciende. Respuesta: Verifica que esté conectada a la corriente...
        - Consulta: No puedo conectar al WiFi. Respuesta: Intenta olvidar la red y reconectar...
        
        Proporciona soluciones prácticas y paso a paso cuando sea necesario."""
    
    def invoke(self, input_data, config=None):
        query = input_data if isinstance(input_data, str) else input_data.get("query", "")
        
        docs = self.vectorstore.similarity_search(query, k=4)
        context = "\n\n".join([d.page_content for d in docs])
        
        prompt = ChatPromptTemplate.from_messages([
            ("system", self.system_prompt),
            ("human", f"Contexto:\n{context}\n\nConsulta: {query}")
        ])
        
        chain = prompt | self.llm
        return {"response": chain.invoke({}).content, "context": context}

class FinanceAgent(Runnable):
    """Agente de Finanzas con RAG."""
    
    def __init__(self, llm, vectorstore):
        self.llm = llm
        self.vectorstore = vectorstore
        
        self.system_prompt = """Eres el agente de Finanzas de la empresa.
        Tu objetivo es asistir con consultas sobre temas financieros y presupuestales.
        
        EJEMPLOS:
        - Consulta: Cómo solicito reembolso? Respuesta: Presenta tu factura con...
        - Consulta: Cuál es el presupuesto de mi área? Respuesta: Consulta en el portal...
        
        Sé preciso con números y fechas."""
    
    def invoke(self, input_data, config=None):
        query = input_data if isinstance(input_data, str) else input_data.get("query", "")
        
        docs = self.vectorstore.similarity_search(query, k=4)
        context = "\n\n".join([d.page_content for d in docs])
        
        prompt = ChatPromptTemplate.from_messages([
            ("system", self.system_prompt),
            ("human", f"Contexto:\n{context}\n\nConsulta: {query}")
        ])
        
        chain = prompt | self.llm
        return {"response": chain.invoke({}).content, "context": context}

# Crear agentes
rrh_agent = RRHAgent(llm, hr_vectorstore)
it_agent = ITAgent(llm, tech_vectorstore)
finance_agent = FinanceAgent(llm, finance_vectorstore)

print("✅ Agentes RRH, IT y Finance creados")

---

## 4. Orquestador y Enrutamiento

In [ ]:
from typing import Dict, List, Any, TypedDict
from langgraph.graph import StateGraph, END

class GraphState(TypedDict):
    query: str
    classification: Dict[str, Any]
    agent_results: List[Dict]
    final_response: str

class IntentRouter:
    """Clasificador de intención del usuario."""
    
    def __init__(self, llm):
        self.llm = llm
        self.categories = ["RRH", "IT", "FINANCE"]
    
    def classify(self, query: str) -> Dict[str, Any]:
        prompt = f"""Clasifica esta consulta en una o más de estas categorías: {', '.join(self.categories)}.
        
        Responde en formato JSON:
        {{"categories": ["RRH", "IT"], "reason": "explicación breve"}}
        
        Consulta: {query}"""
        
        response = self.llm.invoke(prompt)
        try:
            import json
            result = json.loads(response.content)
            return result
        except:
            return {"categories": ["RRH"], "reason": "Default fallback"}

router = IntentRouter(llm)
print("✅ Router de intención configurado")

In [ ]:
def create_support_graph(rrh_agent, it_agent, finance_agent, router):
    """Crea el grafo de soporte multi-agente."""
    
    workflow = StateGraph(GraphState)
    
    def classify_node(state):
        query = state["query"]
        classification = router.classify(query)
        return {"classification": classification}
    
    def delegate_node(state):
        query = state["query"]
        categories = state["classification"].get("categories", [])
        
        results = []
        
        if "RRH" in categories:
            result = rrh_agent.invoke(query)
            results.append({"agent": "RRH", "response": result["response"]})
        
        if "IT" in categories:
            result = it_agent.invoke(query)
            results.append({"agent": "IT", "response": result["response"]})
        
        if "FINANCE" in categories:
            result = finance_agent.invoke(query)
            results.append({"agent": "FINANCE", "response": result["response"]})
        
        # Default to RRH if no category matches
        if not results:
            result = rrh_agent.invoke(query)
            results.append({"agent": "RRH", "response": result["response"]})
        
        return {"agent_results": results}
    
    def consolidate_node(state):
        results = state["agent_results"]
        if not results:
            return {"final_response": "Lo siento, no pude procesar tu consulta."}
        
        response_parts = []
        for r in results:
            response_parts.append(f"**{r['agent']}**:\n{r['response']}")
        
        return {"final_response": "\n\n".join(response_parts)}
    
    # Agregar nodos
    workflow.add_node("classify", classify_node)
    workflow.add_node("delegate", delegate_node)
    workflow.add_node("consolidate", consolidate_node)
    
    # Definir flujo
    workflow.set_entry_point("classify")
    workflow.add_edge("classify", "delegate")
    workflow.add_edge("delegate", "consolidate")
    workflow.add_edge("consolidate", END)
    
    return workflow.compile()

# Compilar grafo
graph = create_support_graph(rrh_agent, it_agent, finance_agent, router)
print("✅ Grafo LangGraph creado")

---

## 5. Pruebas y Ejemplos

In [ ]:
import json

# Cargar queries de prueba
with open("test_queries.json", "r") as f:
    test_data = json.load(f)

test_queries = test_data["test_queries"]
multi_topic_queries = test_data["multi_topic_queries"]

print(f"📋 {len(test_queries)} queries de prueba disponibles")
print(f"📋 {len(multi_topic_queries)} queries multi-tópico disponibles")

In [ ]:
# Ejemplo 1: Consulta de RRH
print("=" * 60)
print("EJEMPLO 1: Consulta de RRH")
print("=" * 60)

query = test_queries[0]["query"]
print(f"📝 Consulta: {query}")
print(f"🎯 Categoría esperada: {test_queries[0]['expected_category']}")

result = graph.invoke({"query": query})
print(f"\n✅ Respuesta:\n{result['final_response'][:500]}")

In [ ]:
# Ejemplo 2: Consulta de IT
print("=" * 60)
print("EJEMPLO 2: Consulta de IT")
print("=" * 60)

query = test_queries[2]["query"]
print(f"📝 Consulta: {query}")

result = graph.invoke({"query": query})
print(f"\n✅ Respuesta:\n{result['final_response'][:500]}")

In [ ]:
# Ejemplo 3: Consulta multi-tópico
print("=" * 60)
print("EJEMPLO 3: Consulta Multi-tópico")
print("=" * 60)

query = multi_topic_queries[0]["query"]
print(f"📝 Consulta: {query}")
print(f"🎯 Categorías esperadas: {multi_topic_queries[0]['expected_categories']}")

result = graph.invoke({"query": query})
print(f"\n✅ Respuesta:\n{result['final_response'][:700]}")

---

## 6. Integración con Langfuse

In [ ]:
from langfuse import Langfuse
from datetime import datetime

class LangfuseTracer:
    """Tracer para Langfuse."""
    
    def __init__(self):
        self.client = None
        self.current_span = None
        
        public_key = os.getenv("LANGFUSE_PUBLIC_KEY")
        secret_key = os.getenv("LANGFUSE_SECRET_KEY")
        host = os.getenv("LANGFUSE_HOST", "https://cloud.langfuse.com")
        
        if public_key and secret_key:
            self.client = Langfuse(
                public_key=public_key,
                secret_key=secret_key,
                host=host
            )
            print("✅ Langfuse conectado")
        else:
            print("⚠️ Langfuse no configurado (configura .env)")
    
    def start_trace(self, query):
        if not self.client:
            return
        self.current_span = self.client.start_observation(
            name="support_query",
            input={"query": query}
        )
    
    def log_classification(self, categories):
        if not self.current_span:
            return
        self.current_span.start_observation(
            name="classification",
            metadata={"categories": categories}
        )
    
    def log_agent(self, agent_name, query, response):
        if not self.current_span:
            return
        span = self.current_span.start_observation(
            name=f"agent_{agent_name}",
            input={"query": query},
            output={"response": response[:500]}
        )
        span.end()
    
    def end_trace(self, final_response):
        if not self.current_span:
            return
        self.current_span.end()
    
    def score(self, name, value):
        if not self.current_span:
            return
        self.current_span.score(name=name, value=value)

# Inicializar tracer
tracer = LangfuseTracer()

In [ ]:
def run_with_tracing(query):
    """Ejecuta el grafo con tracing."""
    tracer.start_trace(query)
    
    # Clasificación
    classification = router.classify(query)
    tracer.log_classification(classification.get("categories", []))
    
    # Ejecución de agentes
    categories = classification.get("categories", [])
    
    results = []
    
    if "RRH" in categories:
        result = rrh_agent.invoke(query)
        results.append({"agent": "RRH", "response": result["response"]})
        tracer.log_agent("RRH", query, result["response"])
    
    if "IT" in categories:
        result = it_agent.invoke(query)
        results.append({"agent": "IT", "response": result["response"]})
        tracer.log_agent("IT", query, result["response"])
    
    if "FINANCE" in categories:
        result = finance_agent.invoke(query)
        results.append({"agent": "FINANCE", "response": result["response"]})
        tracer.log_agent("FINANCE", query, result["response"])
    
    # Consolidar
    response_parts = [f"**{r['agent']}**:\n{r['response']}" for r in results]
    final_response = "\n\n".join(response_parts)
    
    tracer.end_trace(final_response)
    
    return final_response

print("✅ Función de ejecución con tracing definida")

In [ ]:
# Ejecutar con tracing
print("=" * 60)
print("EJEMPLO CON LANGFUSE TRACING")
print("=" * 60)

test_query = "Cuántos días de vacaciones me corresponden con 4 años de antigüedad?"
response = run_with_tracing(test_query)

print(f"\n📝 Consulta: {test_query}")
print(f"\n✅ Respuesta:\n{response[:400]}")
print("\n🌐 Revisa el trace en el dashboard de Langfuse")

---

## Evaluator Agent (BONUS)

In [ ]:
class ResponseEvaluator:
    """Evaluador automatizado de respuestas usando Langfuse Score API."""
    
    def __init__(self, llm):
        self.llm = llm
    
    def evaluate(self, query: str, response: str) -> dict:
        """Evalúa una respuesta en tres dimensiones."""
        
        eval_prompt = f"""Evalúa la siguiente respuesta del asistente.
        
        CONSULTA: {query}
        RESPUESTA: {response[:500]}
        
        Evalúa en escala 0-1:
        - relevance: Qué tan pertinente es la respuesta?
        - completeness: Qué tan completa es la información?
        - accuracy: Qué tan exacta es la información?
        
        Responde solo con: relevance:X.X completeness:X.X accuracy:X.X"""
        
        result = self.llm.invoke(eval_prompt)
        
        # Parse scores
        scores = {"relevance": 0.5, "completeness": 0.5, "accuracy": 0.5}
        for part in result.content.lower().split():
            if "relevance" in part:
                try:
                    scores["relevance"] = float(part.split(":")[1])
                except:
                    pass
            elif "completeness" in part:
                try:
                    scores["completeness"] = float(part.split(":")[1])
                except:
                    pass
            elif "accuracy" in part:
                try:
                    scores["accuracy"] = float(part.split(":")[1])
                except:
                    pass
        
        scores["overall"] = (scores["relevance"] + scores["completeness"] + scores["accuracy"]) / 3
        
        return scores
    
    def get_report(self, scores: dict) -> str:
        """Genera un reporte formateado."""
        status = "✅ BUENA" if scores["overall"] >= 0.7 else "⚠️ REGULAR" if scores["overall"] >= 0.5 else "❌ MALA"
        return f"""
        === REPORTE DE CALIDAD ===
        Overall: {scores['overall']:.2f} {status}
        - Relevancia: {scores['relevance']:.2f}
        - Completitud: {scores['completeness']:.2f}
        - Exactitud: {scores['accuracy']:.2f}
        """

evaluator = ResponseEvaluator(llm)
print("✅ Evaluador configurado")

In [ ]:
# Ejemplo de evaluación
print("=" * 60)
print("EVALUACIÓN DE RESPUESTAS")
print("=" * 60)

test_query = "Cómo solicito reembolso de gastos?"
response = run_with_tracing(test_query)

scores = evaluator.evaluate(test_query, response)
print(evaluator.get_report(scores))

# Score en Langfuse
if tracer.client:
    tracer.score("response_quality", scores["overall"])

---

## Resumen

Este notebook implementa:

1. **Carga de documentos** desde `data/hr_docs/`, `data/tech_docs/`, `data/finance_docs/`
2. **Vector stores** con ChromaDB y sentence-transformers
3. **3 agentes especializados** (RRH, IT, Finance) con RAG y few-shot prompts
4. **Orquestador** con LangGraph para routing condicional
5. **15+ queries de prueba** en `test_queries.json`
6. **Langfuse integration** para tracing y scoring
7. **Evaluator agent** para métricas de calidad

### Para ejecutar:
- Configurar `.env` con API keys
- Ejecutar celdas en orden
- Consultar dashboard de Langfuse para traces